# 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
# %%capture
# !pip install pandas
# !pip install seaborn
# !pip install numpy
# !pip install scikit-learn
# !pip install matplotlib
# !pip install scipy
# !pip install torch
# !pip install xgboost
# !pip install lightgbm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Any
from pandas.core.frame import DataFrame
from pandas import Series
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler, PowerTransformer, StandardScaler
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import os
from datetime import timedelta

# XGBOOST
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

# LSTM & GRU
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

#LGBM
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV

In [ ]:
os.makedirs("stage2_data", exist_ok=True)
os.makedirs("stage2_data/actual_prices", exist_ok=True)
os.makedirs("stage2_data/predicted_prices", exist_ok=True)
os.makedirs("stage2_data/adjusted_prices", exist_ok=True)

In [ ]:
data_dir = "/kaggle/input/final-dataflow-2025"
stock_paths = {
    file.split('_')[-1].replace('.csv', ''): os.path.join(data_dir, file)
    for file in os.listdir(data_dir)
    if file.endswith('.csv')
}

stock_paths

# 1. Data Cleaning

In [ ]:
def clean_data(df: DataFrame) -> DataFrame:
    remove_features: list[str] = ['symbol', 'open_kVND'] # 'high_kVND', 'low_kVND'
    df: DataFrame = df.drop(columns=remove_features, errors='ignore')

    return df

stock_dfs = {
    stock_name: clean_data(pd.read_csv(stock_path)) for stock_name, stock_path in stock_paths.items()
}

# 2. Feature Engineering & Feature Selection

In [ ]:
for stock_name, df in stock_dfs.items():
    df['price_shift_60'] = df['close_kVND'].shift(-365) # adjust to 30 60 90 180 365
    df['return_60'] = (df['price_shift_60'] - df['close_kVND'])*100/df['close_kVND']
    stock_dfs[stock_name] = df.dropna(subset=['price_shift_60'], axis=0)

In [ ]:
for stock_name, df in stock_dfs.items():
  if not isinstance(df.index, pd.DatetimeIndex):
    if 'time' in df.columns:
      df['time'] = pd.to_datetime(df['time'])
      df.set_index('time', inplace=True)

In [ ]:
selected_features_per_stock = {}

required_features = {'sma_7_kVND', 'sma_14_kVND', 'sma_21_kVND', 'rsi_14', 'out_macd_hist_kVND', 'stoch_rsi_14',
                     'bbandsupper_kVND', 'bbandslower_kVND', 'daily_liquidity_kVND', 'obv_kVND', 'adx_14', 'atr_14_kVND',
                     'wma_7_kVND', 'wma_50_kVND', 'low_kVND', 'high_kVND', 'price_to_earning', 'price_to_book'}

for stock_name, df in stock_dfs.items():
    features = df.drop(columns=['price_shift_60']).columns
    label = df['price_shift_60']
    
    selected_features = set(required_features)

    for feature in features:
        if df[feature].isna().any() or df[feature].std() == 0:
            continue

        corr, p_value = pearsonr(df[feature], label)
        if abs(corr) > 0.1 and p_value < 0.05:
            selected_features.add(feature)
    
    print(f"{stock_name}: Selected {len(selected_features)} features out of {len(features)}")
    selected_features_per_stock[stock_name] = selected_features

shared_selected_features = set.intersection(*selected_features_per_stock.values())
shared_selected_features.update(required_features)  # Ensure required indicators are always included

print(f"Shared features across all stocks: {len(shared_selected_features)} features")

for stock_name in stock_dfs.keys():
    stock_dfs[stock_name] = stock_dfs[stock_name][list(shared_selected_features) + ['price_shift_60']]

In [ ]:
len(shared_selected_features)

In [ ]:
def create_train_df_per_stock(features: np.array, label: np.array, lookback: int) -> tuple:
    X, y = [], []
    for i in range(lookback, len(features)):  
        X.append(features[i-lookback:i])  
        y.append(label[i])
    return np.array(X), np.array(y)


label_scalers = {
    stock_name: Pipeline([
        ('minmax_scaler', MinMaxScaler()),
        ('power_transformer', PowerTransformer())
    ]) for stock_name in stock_dfs
}

feature_scalers = {
    stock_name: StandardScaler() for stock_name in stock_dfs
}

datasets = {
    stock_name: {"X_train": None, "X_test": None, "y_train": None, "y_test": None} for stock_name in stock_dfs
}

for stock_name, df in stock_dfs.items():
    lookback = 150

    X = df.drop(["price_shift_60"], axis=1)
    y = df['price_shift_60'].values

    y = label_scalers[stock_name].fit_transform(y.reshape(-1, 1))
    X = feature_scalers[stock_name].fit_transform(X) 
    X, y = create_train_df_per_stock(X, y, lookback) # X shape (2347, 150, 84) | y shape (2347,)

    split = int(len(y) * 0.85)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    # Store the datasets for each stock
    datasets[stock_name]["X_train"] = X_train
    datasets[stock_name]["X_test"] = X_test
    datasets[stock_name]["y_train"] = y_train
    datasets[stock_name]["y_test"] = y_test
    print(X_train.shape, X_test.shape)

In [ ]:
total_len_train = 0
for stock_name, dataset in datasets.items():
  if dataset["X_train"] is not None:
    total_len_train += len(dataset["X_train"])

print(f"Total length of X_train across all stocks: {total_len_train}")

In [ ]:
def apply_post_processing_rules(predicted_price: float, current_price: float, df: pd.DataFrame, index: int) -> float:
    indicators = {
        'MA_7': df['sma_7_kVND'].iloc[index],
        'MA_14': df['sma_14_kVND'].iloc[index],
        'MA_21': df['sma_21_kVND'].iloc[index],
        'RSI': df['rsi_14'].iloc[index],
        'MACD_bullish': df['out_macd_hist_kVND'].iloc[index] > 0,
        'MACD_bearish': df['out_macd_hist_kVND'].iloc[index] < 0,
        'MACD_divergence': False,
        'Stochastic': df['stoch_rsi_14'].iloc[index],
        'BB_upper': df['bbandsupper_kVND'].iloc[index],
        'BB_lower': df['bbandslower_kVND'].iloc[index],
        'volume_trend': 'strong' if df['daily_liquidity_kVND'].iloc[index] > df['daily_liquidity_kVND'].iloc[index-1] else 'weak',
        'OBV_confirm': df['obv_kVND'].iloc[index] > df['obv_kVND'].iloc[index-1],
        'ADX': df['adx_14'].iloc[index],
        'ATR': df['atr_14_kVND'].iloc[index],
        'time_series_RSI': df['rsi_14'].iloc[index-5:index].tolist(),
        'slope_short': df['wma_7_kVND'].iloc[index] - df['wma_7_kVND'].iloc[index-1],
        'slope_long': df['wma_50_kVND'].iloc[index] - df['wma_50_kVND'].iloc[index-1],
        'support_level': df['low_kVND'].iloc[index-5:index].min(),
        'resistance_level': df['high_kVND'].iloc[index-5:index].max(),
        'PE': df['price_to_earning'].iloc[index],
        'PB': df['price_to_book'].iloc[index],
        'PE_industry': df['price_to_earning'].mean(),
        'PB_industry': df['price_to_book'].mean(),
        'consensus_down': sum([df[col].iloc[index] < df[col].iloc[index-1] for col in ['ema_12_kVND', 'sma_21_kVND', 'wma_14_kVND']]),
        'consensus_up': sum([df[col].iloc[index] > df[col].iloc[index-1] for col in ['ema_12_kVND', 'sma_21_kVND', 'wma_14_kVND']])
    }

    adjusted_price = predicted_price

     # 1. Quy tắc điều chỉnh theo ‘Overbought’
    if (predicted_price > current_price and 
        indicators['MA_7'] > indicators['MA_14'] > indicators['MA_21'] and 
        indicators['RSI'] > 70):
        adjusted_price *= 0.98
        
    # 2. Quy tắc điều chỉnh theo ‘Oversold’
    if (predicted_price < current_price and 
        indicators['MA_7'] < indicators['MA_14'] < indicators['MA_21'] and 
        indicators['RSI'] < 30):
        adjusted_price *= 1.02
        
    # 3. Quy tắc xác nhận xu hướng ‘Bullish’
    if (predicted_price > current_price and 
        indicators['MA_7'] > indicators['MA_14'] > indicators['MA_21'] and 
        50 <= indicators['RSI'] <= 70 and 
        indicators.get('MACD_bullish', False)):
        adjusted_price *= 1.01
        
    # 4. Quy tắc xác nhận xu hướng ‘Bearish’
    if (predicted_price < current_price and 
        indicators['MA_7'] < indicators['MA_14'] < indicators['MA_21'] and 
        30 <= indicators['RSI'] <= 50 and 
        indicators.get('MACD_bearish', False)):
        adjusted_price *= 0.99
        
    # 5. Quy tắc cho trạng thái ‘Neutral’
    if (abs(predicted_price - current_price) / current_price < 0.005 and 
        abs(indicators['MA_7'] - indicators['MA_14']) < 0.5 and 
        abs(indicators['MA_14'] - indicators['MA_21']) < 0.5 and 
        abs(indicators['RSI'] - 50) < 5):
        adjusted_price *= 1.0
        
    # 6. Quy tắc xử lí trường hợp tín hiệu ‘Mixed’
    if ((indicators['MA_7'] > indicators['MA_14'] and not (indicators['MA_14'] > indicators['MA_21'])) or 
        (45 <= indicators['RSI'] <= 55)):
        if predicted_price > current_price:
            adjusted_price *= 0.995
        elif predicted_price < current_price:
            adjusted_price *= 1.005
            
    # 7. Luật điều chỉnh dựa trên Bollinger Bands (BB)
    if (predicted_price > current_price and 
        predicted_price >= indicators.get('BB_upper', float('inf')) and 
        (indicators['RSI'] > 70 or indicators.get('Stochastic', 0) > 80)):
        adjusted_price *= 0.975
        
    # 8. Luật MACD và Divergence
    if (predicted_price > current_price and 
        not indicators.get('MACD_bullish', True) and 
        indicators.get('MACD_divergence', False) and 
        indicators['ADX'] < 20):
        adjusted_price *= 0.98
    
    # 9. Luật sử dụng Stochastic Oscillator
    if predicted_price > current_price and indicators.get('Stochastic', 0) > 80:
        adjusted_price *= 0.98
    if predicted_price < current_price and indicators.get('Stochastic', 0) < 20:
        adjusted_price *= 1.02

    # 10. Luật kết hợp Volume và OBV
    if predicted_price > current_price and indicators.get('volume_trend', '') == 'weak':
        adjusted_price *= 0.99
    if predicted_price < current_price and indicators.get('volume_trend', '') == 'strong':
        adjusted_price *= 1.01

    # 11. Luật sử dụng ADX để đánh giá sức mạnh xu hướng
    if predicted_price > current_price and indicators['ADX'] < 25:
        adjusted_price *= 0.99
    if predicted_price < current_price and indicators['ADX'] < 25:
        adjusted_price *= 1.01

    # 12. Luật dựa trên ATR (Average True Range)
    if predicted_price > current_price and indicators['ATR'] < 0.5:
        adjusted_price *= 0.98
    if predicted_price < current_price and indicators['ATR'] < 0.5:
        adjusted_price *= 0.98
    
    # 13. Luật kết hợp nhiều chỉ báo (consensus)
    if predicted_price > current_price and indicators.get('consensus_down', 0) >= 5:
        adjusted_price *= 0.97
    if predicted_price < current_price and indicators.get('consensus_up', 0) >= 5:
        adjusted_price *= 1.02

    # 14. Luật điều chỉnh theo Time-Weighted Factor
    ts_rsi = indicators.get('time_series_RSI', [])
    if len(ts_rsi) >= 3:
        recent_rsi = ts_rsi[-3:]
        if all(rsi > 70 for rsi in recent_rsi):
            adjusted_price *= 0.97
        elif all(rsi < 30 for rsi in recent_rsi):
            adjusted_price *= 1.03

    # 15. Luật đa khung thời gian
    if 'slope_short' in indicators and 'slope_long' in indicators:
        if predicted_price > current_price and indicators['slope_long'] < 0 and indicators['slope_short'] > 0:
            adjusted_price *= 0.98
        elif predicted_price < current_price and indicators['slope_long'] > 0 and indicators['slope_short'] < 0:
            adjusted_price *= 1.01
            
    # 16. Luật hỗ trợ - kháng cự
    if 'support_level' in indicators and 'resistance_level' in indicators:
        if abs(predicted_price - indicators['resistance_level']) / indicators['resistance_level'] < 0.01:
            adjusted_price *= 0.975
        if abs(predicted_price - indicators['support_level']) / indicators['support_level'] < 0.01:
            adjusted_price *= 1.015

    # 17. Overvaluation
    if predicted_price > current_price and (indicators.get('PE', 0) > 25 or indicators.get('PB', 0) > 3):
        if indicators.get('PE', 0) > indicators.get('PE_industry', 25) or indicators.get('PB', 0) > indicators.get('PB_industry', 3):
            adjusted_price *= 0.975

    # 18. Undervaluation
    if predicted_price < current_price and (indicators.get('PE', float('inf')) < 15 or indicators.get('PB', float('inf')) < 1):
        if indicators.get('PE', float('inf')) < indicators.get('PE_industry', 15) or indicators.get('PB', float('inf')) < indicators.get('PB_industry', 1):
            adjusted_price *= 1.02
            
    return adjusted_price

# 4. LSTM Finetuning

In [ ]:
# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

BATCH_SIZE = 64

In [ ]:
X_train = np.concatenate([d["X_train"] for d in datasets.values() if d["X_train"] is not None])
y_train = np.concatenate([d["y_train"] for d in datasets.values() if d["y_train"] is not None])
X_test = np.concatenate([d["X_test"] for d in datasets.values() if d["X_test"] is not None])
y_test = np.concatenate([d["y_test"] for d in datasets.values() if d["y_test"] is not None])

print("Combined X_train shape:", X_train.shape)
print("Combined y_train shape:", y_train.shape)
print(X_test.shape, y_test.shape)

# Prepare data loaders
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_split = int(0.8 * len(train_dataset))
train_subset, val_subset = torch.utils.data.random_split(train_dataset, [val_split, len(train_dataset) - val_split])
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=50, patience=10):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)  # reduced learning rate

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)
        print(f"Epoch {epoch+1}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    model.load_state_dict(torch.load('best_model.pt'))
    return model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, 120, batch_first=True)
        self.lstm2 = nn.LSTM(120, 64, batch_first=True)
        self.lstm3 = nn.LSTM(64, 32, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x, _ = self.lstm(x)
        x, _ = self.lstm2(x)
        x, (hn, _) = self.lstm3(x)
        x = hn.squeeze(0)
        return self.fc(x)

print("\nTraining LSTM model:")
lstm_model = LSTMModel(X_train.shape[2]).to(device)
lstm_model = train_model(lstm_model, train_loader, val_loader)

lstm_model.eval()
with torch.no_grad():
    lstm_preds = lstm_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

print("LSTM RMSE:", np.sqrt(mean_squared_error(y_test, lstm_preds)))
print("LSTM MAPE:", mean_absolute_percentage_error(y_test, lstm_preds))
print("LSTM R2:", r2_score(y_test, lstm_preds))

## 4.1 LSTM Prediction

In [ ]:
lookback = 150

predictions = {}
metrics = {}

for stock_name, df in stock_dfs.items():
    df = df.sort_index()
    df = df[list(shared_selected_features) + ['price_shift_60']]
    split = int(len(df) * 0.85)
    test_df = df.iloc[split:]
    extended_test_df = df.iloc[split - lookback:]

    if len(extended_test_df) < lookback:
        print(f"Not enough test data for stock {stock_name}")
        continue

    X_raw = extended_test_df.drop("price_shift_60", axis=1)
    y_raw = extended_test_df["price_shift_60"]
    X_scaled = feature_scalers[stock_name].transform(X_raw)
    y_scaled = label_scalers[stock_name].transform(y_raw.values.reshape(-1, 1))
    X_test_all, y_test_all = create_train_df_per_stock(X_scaled, y_scaled, lookback)
    window_dates = extended_test_df.index[lookback:]
    mask = window_dates.isin(test_df.index)
    X_test = X_test_all[mask]
    y_test = y_test_all[mask]
    dates_selected = window_dates[mask]

    if X_test.ndim == 1:
        X_test = X_test.reshape(1, lookback, -1)
    elif X_test.ndim == 2:
        X_test = np.expand_dims(X_test, axis=0)

    lstm_model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_pred_scaled = lstm_model(X_tensor).cpu().numpy()

    y_pred = label_scalers[stock_name].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1)).flatten()

    adjusted_preds = []
    for date, pred_price in zip(dates_selected, y_pred):
        if date in df.index:
            idx = df.index.get_loc(date)
            current_price = df['close_kVND'].iloc[idx]
            adjusted_price = apply_post_processing_rules(pred_price, current_price, df, idx)
            adjusted_preds.append(adjusted_price)
        else:
            adjusted_preds.append(pred_price)

    results_df = pd.DataFrame({
        "Date": dates_selected,
        "Predicted": y_pred,
        "Adjusted_Predicted": adjusted_preds,
        "Actual": y_true
    })
    results_df.set_index("Date", inplace=True)

    predictions[stock_name] = results_df

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    metrics[stock_name] = {"RMSE": rmse, "MAPE": mape, "R2": r2}

# Clear formatted printing
for stock, df_results in predictions.items():
    print(f"Predictions for {stock}:")
    print(df_results)
    print("\n")

# Print performance metrics clearly
print("Performance Metrics:")
for stock, m in metrics.items():
    print(f"{stock}: RMSE={m['RMSE']:.4f}, MAPE={m['MAPE']:.4f}, R2={m['R2']:.4f}")

In [ ]:
for stock, df_results in predictions.items():
    pred_rmse = np.sqrt(mean_squared_error(df_results["Actual"], df_results["Predicted"]))
    pred_mape = mean_absolute_percentage_error(df_results["Actual"], df_results["Predicted"])
    pred_r2 = r2_score(df_results["Actual"], df_results["Predicted"])

    adj_rmse = np.sqrt(mean_squared_error(df_results["Actual"], df_results["Adjusted_Predicted"]))
    adj_mape = mean_absolute_percentage_error(df_results["Actual"], df_results["Adjusted_Predicted"])
    adj_r2 = r2_score(df_results["Actual"], df_results["Adjusted_Predicted"])

    print(f"Metrics for {stock}:")
    print(f"  Predicted vs Actual -> RMSE: {pred_rmse:.4f}, MAPE: {pred_mape:.4f}, R2: {pred_r2:.4f}")
    print(f"  Adjusted vs Actual  -> RMSE: {adj_rmse:.4f}, MAPE: {adj_mape:.4f}, R2: {adj_r2:.4f}\n")

In [ ]:
actual_prices_df = pd.DataFrame({stock: df["Actual"] for stock, df in predictions.items()})
predicted_prices_df = pd.DataFrame({stock: df["Predicted"] for stock, df in predictions.items()})
adjusted_prices_df = pd.DataFrame({stock: df["Adjusted_Predicted"] for stock, df in predictions.items()})

actual_prices_df.to_csv("/kaggle/working/stage2_data/actual_prices/LSTM_365_Days_Actual_Prices.csv")
predicted_prices_df.to_csv("/kaggle/working/stage2_data/predicted_prices/LSTM_365_Days_Predicted_Prices.csv")
adjusted_prices_df.to_csv("/kaggle/working/stage2_data/adjusted_prices/LSTM_365_Days_Adjusted_Prices.csv")

print("Files saved successfully.")

# 5. GRU Finetuning

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_size, 64, batch_first=True)
        self.gru2 = nn.GRU(64, 32, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        x, _ = self.gru(x)
        x, hn = self.gru2(x)  # fixed unpacking
        x = hn.squeeze(0)
        return self.fc(x)

print("\nTraining GRU model:")
gru_model = GRUModel(X_train.shape[2]).to(device)
gru_model = train_model(gru_model, train_loader, val_loader)

# Evaluate GRU
gru_model.eval()
with torch.no_grad():
    gru_preds = gru_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

print("GRU RMSE:", np.sqrt(mean_squared_error(y_test, gru_preds)))
print("GRU MAPE:", mean_absolute_percentage_error(y_test, gru_preds))
print("GRU R2:", r2_score(y_test, gru_preds))

## 5.1 GRU Predictions

In [ ]:
lookback = 150
predictions = {}
metrics = {}

for stock_name, df in stock_dfs.items():
    df = df.sort_index()
    df = df[list(shared_selected_features) + ['price_shift_60']]
    split = int(len(df) * 0.85)
    test_df = df.iloc[split:]
    extended_test_df = df.iloc[split - lookback:]

    if len(extended_test_df) < lookback:
        print(f"Not enough test data for stock {stock_name}")
        continue

    X_raw = extended_test_df.drop("price_shift_60", axis=1)
    y_raw = extended_test_df["price_shift_60"]
    X_scaled = feature_scalers[stock_name].transform(X_raw)
    y_scaled = label_scalers[stock_name].transform(y_raw.values.reshape(-1, 1))
    X_test_all, y_test_all = create_train_df_per_stock(X_scaled, y_scaled, lookback)
    window_dates = extended_test_df.index[lookback:]
    mask = window_dates.isin(test_df.index)
    X_test = X_test_all[mask]
    y_test = y_test_all[mask]
    dates_selected = window_dates[mask]

    if X_test.ndim == 1:
        X_test = X_test.reshape(1, lookback, -1)
    elif X_test.ndim == 2:
        X_test = np.expand_dims(X_test, axis=0)

    gru_model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_pred_scaled = gru_model(X_tensor).cpu().numpy()

    y_pred = label_scalers[stock_name].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1)).flatten()

    adjusted_preds = []
    for date, pred_price in zip(dates_selected, y_pred):
        if date in df.index:
            idx = df.index.get_loc(date)
            current_price = df['close_kVND'].iloc[idx]
            adjusted_price = apply_post_processing_rules(pred_price, current_price, df, idx)
            adjusted_preds.append(adjusted_price)
        else:
            adjusted_preds.append(pred_price)

    results_df = pd.DataFrame({
        "Date": dates_selected,
        "Predicted": y_pred,
        "Adjusted_Predicted": adjusted_preds,
        "Actual": y_true
    })
    results_df.set_index("Date", inplace=True)

    predictions[stock_name] = results_df

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    metrics[stock_name] = {"RMSE": rmse, "MAPE": mape, "R2": r2}

for stock, df_results in predictions.items():
    print(f"Predictions for {stock}:")
    print(df_results)
    print("\n")

print("Performance Metrics (GRU Predictions):")
for stock, m in metrics.items():
    print(f"{stock}: RMSE={m['RMSE']:.4f}, MAPE={m['MAPE']:.4f}, R2={m['R2']:.4f}")

In [ ]:
print("\nAdditional Metrics (Adjusted Predictions) for GRU:")
for stock, df_results in predictions.items():
    adj_rmse = np.sqrt(mean_squared_error(df_results["Actual"], df_results["Adjusted_Predicted"]))
    adj_mape = mean_absolute_percentage_error(df_results["Actual"], df_results["Adjusted_Predicted"])
    adj_r2 = r2_score(df_results["Actual"], df_results["Adjusted_Predicted"])

    print(f"{stock}: RMSE={adj_rmse:.4f}, MAPE={adj_mape:.4f}, R2={adj_r2:.4f}")

In [ ]:
# Save results to CSV
actual_prices_df = pd.DataFrame({stock: df["Actual"] for stock, df in predictions.items()})
predicted_prices_df = pd.DataFrame({stock: df["Predicted"] for stock, df in predictions.items()})
adjusted_prices_df = pd.DataFrame({stock: df["Adjusted_Predicted"] for stock, df in predictions.items()})

actual_prices_df.to_csv("/kaggle/working/stage2_data/actual_prices/GRU_365_Days_Actual_Prices.csv")
predicted_prices_df.to_csv("/kaggle/working/stage2_data/predicted_prices/GRU_365_Days_Predicted_Prices.csv")
adjusted_prices_df.to_csv("/kaggle/working/stage2_data/adjusted_prices/GRU_365_Days_Adjusted_Prices.csv")

print("\nGRU model pipeline completed. Files saved successfully.")

# 8. Bi-LSTM-Attention Finetuning

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, input_size, hidden_size=64):
        super(CNN_BiLSTM_Attention, self).__init__()
        self.conv1 = nn.Conv1d(in_channels=input_size, out_channels=64, kernel_size=3, padding=1)
        self.dropout_conv = nn.Dropout(0.2)

        self.bi_lstm_1 = nn.LSTM(input_size=64, hidden_size=64, batch_first=True, bidirectional=True)
        self.bi_lstm_2 = nn.LSTM(input_size=128, hidden_size=64, batch_first=True, bidirectional=True)

        self.attention = nn.MultiheadAttention(embed_dim=128, num_heads=4, batch_first=True)

        self.fc1 = nn.Linear(128, 64)
        self.dropout_fc = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Input: (B, T, C) → permute for Conv1D: (B, C, T)
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv1(x))
        x = self.dropout_conv(x)

        # Back to (B, T, C) for LSTM
        x = x.permute(0, 2, 1)

        x, _ = self.bi_lstm_1(x)
        x, _ = self.bi_lstm_2(x)

        # Attention: input must be (B, T, E)
        attn_output, _ = self.attention(x, x, x)  # self-attention

        # Pool the sequence into a single vector (like LSTMModel)
        # We'll use the mean over time dimension
        pooled = attn_output.mean(dim=1)

        x = F.relu(self.fc1(pooled))
        x = self.dropout_fc(x)
        return self.fc2(x)

In [ ]:
bi_model = CNN_BiLSTM_Attention(input_size=X_train.shape[2]).to(device)
bi_model = train_model(bi_model, train_loader, val_loader)

bi_model.eval()
with torch.no_grad():
    preds = bi_model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

print("CNN-BiLSTM-Attn RMSE:", np.sqrt(mean_squared_error(y_test, preds)))
print("CNN-BiLSTM-Attn MAPE:", mean_absolute_percentage_error(y_test, preds))
print("CNN-BiLSTM-Attn R2:", r2_score(y_test, preds))

In [ ]:
lookback = 150

predictions = {}
metrics = {}

for stock_name, df in stock_dfs.items():
    df = df.sort_index()
    df = df[list(shared_selected_features) + ['price_shift_60']]
    split = int(len(df) * 0.85)
    test_df = df.iloc[split:]
    extended_test_df = df.iloc[split - lookback:]

    if len(extended_test_df) < lookback:
        print(f"Not enough test data for stock {stock_name}")
        continue

    X_raw = extended_test_df.drop("price_shift_60", axis=1)
    y_raw = extended_test_df["price_shift_60"]

    X_scaled = feature_scalers[stock_name].transform(X_raw)
    y_scaled = label_scalers[stock_name].transform(y_raw.values.reshape(-1, 1))

    X_test_all, y_test_all = create_train_df_per_stock(X_scaled, y_scaled, lookback)
    window_dates = extended_test_df.index[lookback:]
    mask = window_dates.isin(test_df.index)

    X_test = X_test_all[mask]
    y_test = y_test_all[mask]
    dates_selected = window_dates[mask]

    if X_test.ndim == 1:
        X_test = X_test.reshape(1, lookback, -1)
    elif X_test.ndim == 2:
        X_test = np.expand_dims(X_test, axis=0)

    bi_model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
        y_pred_scaled = bi_model(X_tensor).cpu().numpy()

    y_pred = label_scalers[stock_name].inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1)).flatten()

    adjusted_preds = []
    for date, pred_price in zip(dates_selected, y_pred):
        if date in df.index:
            idx = df.index.get_loc(date)
            current_price = df['close_kVND'].iloc[idx]
            adjusted_price = apply_post_processing_rules(pred_price, current_price, df, idx)
            adjusted_preds.append(adjusted_price)
        else:
            adjusted_preds.append(pred_price)

    results_df = pd.DataFrame({
        "Date": dates_selected,
        "Predicted": y_pred,
        "Adjusted_Predicted": adjusted_preds,
        "Actual": y_true
    })
    results_df.set_index("Date", inplace=True)

    predictions[stock_name] = results_df

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    metrics[stock_name] = {"RMSE": rmse, "MAPE": mape, "R2": r2}

# Clear formatted printing
for stock, df_results in predictions.items():
    print(f"Predictions for {stock}:")
    print(df_results)
    print("\n")

# Print performance metrics clearly
print("Performance Metrics:")
for stock, m in metrics.items():
    print(f"{stock}: RMSE={m['RMSE']:.4f}, MAPE={m['MAPE']:.4f}, R2={m['R2']:.4f}")

In [ ]:
# Print raw and adjusted metrics
print("Performance Metrics:")
for stock, df_results in predictions.items():
    pred_rmse = np.sqrt(mean_squared_error(df_results["Actual"], df_results["Predicted"]))
    pred_mape = mean_absolute_percentage_error(df_results["Actual"], df_results["Predicted"])
    pred_r2 = r2_score(df_results["Actual"], df_results["Predicted"])

    adj_rmse = np.sqrt(mean_squared_error(df_results["Actual"], df_results["Adjusted_Predicted"]))
    adj_mape = mean_absolute_percentage_error(df_results["Actual"], df_results["Adjusted_Predicted"])
    adj_r2 = r2_score(df_results["Actual"], df_results["Adjusted_Predicted"])

    print(f"Metrics for {stock}:")
    print(f"  Predicted vs Actual -> RMSE: {pred_rmse:.4f}, MAPE: {pred_mape:.4f}, R2: {pred_r2:.4f}")
    print(f"  Adjusted vs Actual  -> RMSE: {adj_rmse:.4f}, MAPE: {adj_mape:.4f}, R2: {adj_r2:.4f}\n")

In [ ]:
# Save outputs to CSV
actual_prices_df = pd.DataFrame({stock: df["Actual"] for stock, df in predictions.items()})
predicted_prices_df = pd.DataFrame({stock: df["Predicted"] for stock, df in predictions.items()})
adjusted_prices_df = pd.DataFrame({stock: df["Adjusted_Predicted"] for stock, df in predictions.items()})

actual_prices_df.to_csv("/kaggle/working/stage2_data/actual_prices/CNN_BiLSTM_365_Days_Actual_Prices.csv")
predicted_prices_df.to_csv("/kaggle/working/stage2_data/predicted_prices/CNN_BiLSTM_365_Days_Predicted_Prices.csv")
adjusted_prices_df.to_csv("/kaggle/working/stage2_data/adjusted_prices/CNN_BiLSTM_365_Days_Adjusted_Prices.csv")

print("Files saved successfully.")

# 7. Evaluation / Plotting MAPE, MSE, R^2

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, stock_name, is_torch=True):
    if is_torch:
        model.eval()
        with torch.no_grad():
            y_pred = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()
    else:
        X_flat = X_test.reshape(X_test.shape[0], -1)
        y_pred = model.predict(X_flat)

    y_test_true = label_scalers[stock_name].inverse_transform(y_test.reshape(-1, 1))
    y_pred = label_scalers[stock_name].inverse_transform(y_pred.reshape(-1, 1))

    mape = mean_absolute_percentage_error(y_test_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_true, y_pred))
    r2 = r2_score(y_test_true, y_pred)

    return mape, rmse, r2


metrics_values = []

for stock_name, dataset in datasets.items():
    if dataset["X_test"] is None:
        continue

    X_test = dataset["X_test"]
    y_test = dataset["y_test"]

    # LSTM
    mape_lstm, rmse_lstm, r2_lstm = evaluate_model(lstm_model, X_test, y_test, "LSTM", stock_name)
    metrics_values.append((stock_name, "LSTM", mape_lstm, rmse_lstm, r2_lstm))

    # GRU
    mape_gru, rmse_gru, r2_gru = evaluate_model(gru_model, X_test, y_test, "GRU", stock_name)
    metrics_values.append((stock_name, "GRU", mape_gru, rmse_gru, r2_gru))

    # CNN-BiLSTM-Attention
    mape_bi_model, rmse_bi_model, r2_bi_model = evaluate_model(bi_model, X_test, y_test, "CNN_BiLSTM_Attn", stock_name)
    metrics_values.append((stock_name, "CNN_BiLSTM_Attn", mape_bi_model, rmse_bi_model, r2_bi_model))

# Convert to DataFrame
metrics_df = pd.DataFrame(metrics_values, columns=['Stock', 'Model', 'MAPE', 'RMSE', 'R2'])
os.makedirs("/kaggle/working/stage2_data/figures", exist_ok=True)  # Create folder for figures

# ========== Distribution Plots ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE Distribution Plot
sns.histplot(metrics_df['RMSE'], kde=True, bins=10, ax=axes[0], color='blue')
axes[0].set_title('RMSE Distribution')
axes[0].set_xlabel('RMSE')
axes[0].set_ylabel('Frequency')

# MAPE Distribution Plot
sns.histplot(metrics_df['MAPE'], kde=True, bins=10, ax=axes[1], color='green')
axes[1].set_title('MAPE Distribution')
axes[1].set_xlabel('MAPE')
axes[1].set_ylabel('Frequency')

# R2 Distribution Plot
sns.histplot(metrics_df['R2'], kde=True, bins=10, ax=axes[2], color='red')
axes[2].set_title('R² Distribution')
axes[2].set_xlabel('R²')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig("/kaggle/working/stage2_data/figures/metrics_distribution_plots_365_days.png")
plt.show()

In [ ]:
# ========== Model Comparison Bar Plots ==========
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# RMSE Plot
sns.barplot(x='Stock', y='RMSE', hue='Model', data=metrics_df, ax=axes[0])
axes[0].set_title('RMSE for All Stocks and Models')
axes[0].tick_params(axis='x', rotation=90)

# MAPE Plot
sns.barplot(x='Stock', y='MAPE', hue='Model', data=metrics_df, ax=axes[1])
axes[1].set_title('MAPE for All Stocks and Models')
axes[1].tick_params(axis='x', rotation=90)

# R2 Plot
sns.barplot(x='Stock', y='R2', hue='Model', data=metrics_df, ax=axes[2])
axes[2].set_title('R² for All Stocks and Models')
axes[2].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig("/kaggle/working/stage2_data/figures/metrics_bar_comparison_365_days.png")
plt.show()


In [ ]:
!zip -r stage2_data.zip /kaggle/working/stage2_data

# That's the end for Stage 1